# 据统计分析建议
根据你提供的文件名，我可以看出你的数据库中包含了价格来源、天气数据和资讯新闻数据表，以及一个数据库模式的Excel文件。基于这些信息，我可以为你提供一些数据统计分析的建议。

## 1. 价格趋势分析

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import psycopg2
from datetime import datetime, timedelta
import numpy as np

def connect_to_db():
    """连接到数据库"""
    conn = psycopg2.connect(
        host="localhost",
        database="herb_price_db",
        user="postgres",
        password="your_password"
    )
    return conn

def analyze_price_trends(herb_name, specification, days=90):
    """分析特定药材的价格趋势"""
    conn = connect_to_db()
    
    # 计算日期范围
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)
    
    # 查询价格数据
    query = """
    SELECT 
        recorded_at::date as date, 
        AVG(price::numeric) as avg_price,
        MIN(price::numeric) as min_price,
        MAX(price::numeric) as max_price,
        source
    FROM herb_prices
    WHERE herb_name = %s 
      AND specification = %s
      AND recorded_at >= %s
    GROUP BY recorded_at::date, source
    ORDER BY recorded_at::date
    """
    
    df = pd.read_sql(query, conn, params=(herb_name, specification, start_date))
    
    # 关闭数据库连接
    conn.close()
    
    if df.empty:
        print(f"没有找到{herb_name} {specification}的价格数据")
        return
    
    # 数据分析
    # 1. 计算价格变化率
    df['date'] = pd.to_datetime(df['date'])
    df_pivot = df.pivot_table(index='date', columns='source', values='avg_price')
    df_pivot = df_pivot.fillna(method='ffill')
    
    # 计算每个来源的价格变化率
    for source in df_pivot.columns:
        df_pivot[f'{source}_change'] = df_pivot[source].pct_change() * 100
    
    # 2. 计算价格波动性 (标准差)
    volatility = df.groupby('source')['avg_price'].std()
    
    # 3. 计算价格季节性
    # 添加月份列
    df['month'] = df['date'].dt.month
    seasonal_avg = df.groupby(['month', 'source'])['avg_price'].mean()
    
    # 4. 计算价格趋势 (线性回归)
    from sklearn.linear_model import LinearRegression
    
    trends = {}
    for source in df['source'].unique():
        source_df = df[df['source'] == source]
        if len(source_df) > 1:  # 确保有足够的数据点
            X = np.array(range(len(source_df))).reshape(-1, 1)
            y = source_df['avg_price'].values
            model = LinearRegression()
            model.fit(X, y)
            trends[source] = {
                'slope': model.coef_[0],
                'intercept': model.intercept_
            }
    
    # 输出分析结果
    print(f"\n{herb_name} {specification} 价格分析结果:")
    print(f"分析周期: {start_date.date()} 至 {end_date.date()}")
    print("\n价格统计:")
    print(f"平均价格: {df['avg_price'].mean():.2f}")
    print(f"最高价格: {df['max_price'].max():.2f}")
    print(f"最低价格: {df['min_price'].min():.2f}")
    
    print("\n价格波动性 (标准差):")
    for source, std in volatility.items():
        print(f"{source}: {std:.2f}")
    
    print("\n价格趋势 (斜率):")
    for source, trend in trends.items():
        trend_direction = "上升" if trend['slope'] > 0 else "下降"
        print(f"{source}: {trend_direction}, 斜率={trend['slope']:.4f}")
    
    # 可视化
    plt.figure(figsize=(12, 8))
    
    # 绘制价格趋势
    plt.subplot(2, 1, 1)
    for source in df['source'].unique():
        source_df = df[df['source'] == source]
        plt.plot(source_df['date'], source_df['avg_price'], label=source)
    
    plt.title(f"{herb_name} {specification} 价格趋势")
    plt.xlabel("日期")
    plt.ylabel("价格")
    plt.legend()
    plt.grid(True)
    
    # 绘制价格变化率
    plt.subplot(2, 1, 2)
    for source in df_pivot.columns:
        if not source.endswith('_change'):
            plt.plot(df_pivot.index, df_pivot[f'{source}_change'], label=f"{source}变化率")
    
    plt.title("价格变化率 (%)")
    plt.xlabel("日期")
    plt.ylabel("变化率 (%)")
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(f"{herb_name}_{specification}_price_analysis.png")
    plt.show()
    
    return df

# 示例使用
if __name__ == "__main__":
    analyze_price_trends("当归", "统", 180)

2. 天气与价格相关性分析

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import psycopg2
import seaborn as sns
from datetime import datetime, timedelta
import numpy as np
from scipy.stats import pearsonr

def connect_to_db():
    """连接到数据库"""
    conn = psycopg2.connect(
        host="localhost",
        database="herb_price_db",
        user="postgres",
        password="your_password"
    )
    return conn

def analyze_weather_price_correlation(herb_name, specification, location=None, days=180):
    """分析天气与价格的相关性"""
    conn = connect_to_db()
    
    # 计算日期范围
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)
    
    # 构建查询
    query = """
    SELECT 
        hp.recorded_at::date as date,
        AVG(hp.price::numeric) as avg_price,
        AVG(wd.temperature) as avg_temp,
        AVG(wd.precipitation) as avg_precip,
        AVG(wd.humidity) as avg_humidity,
        hp.location
    FROM herb_prices hp
    JOIN weather_data wd ON hp.location = wd.location AND hp.recorded_at::date = wd.date
    WHERE hp.herb_name = %s 
      AND hp.specification = %s
      AND hp.recorded_at >= %s
    """
    
    params = [herb_name, specification, start_date]
    
    if location:
        query += " AND hp.location = %s"
        params.append(location)
        
    query += """
    GROUP BY hp.recorded_at::date, hp.location
    ORDER BY hp.recorded_at::date
    """
    
    df = pd.read_sql(query, conn, params=params)
    
    # 关闭数据库连接
    conn.close()
    
    if df.empty:
        print(f"没有找到{herb_name} {specification}的相关数据")
        return
    
    # 数据分析
    # 1. 计算相关系数
    correlations = {}
    for location in df['location'].unique():
        loc_df = df[df['location'] == location]
        
        # 温度与价格的相关性
        temp_corr, temp_p = pearsonr(loc_df['avg_temp'].fillna(0), loc_df['avg_price'])
        
        # 降水量与价格的相关性
        precip_corr, precip_p = pearsonr(loc_df['avg_precip'].fillna(0), loc_df['avg_price'])
        
        # 湿度与价格的相关性
        humidity_corr, humidity_p = pearsonr(loc_df['avg_humidity'].fillna(0), loc_df['avg_price'])
        
        correlations[location] = {
            'temperature': {'corr': temp_corr, 'p_value': temp_p},
            'precipitation': {'corr': precip_corr, 'p_value': precip_p},
            'humidity': {'corr': humidity_corr, 'p_value': humidity_p}
        }
    
    # 输出分析结果
    print(f"\n{herb_name} {specification} 天气与价格相关性分析:")
    print(f"分析周期: {start_date.date()} 至 {end_date.date()}")
    
    for location, corrs in correlations.items():
        print(f"\n地区: {location}")
        print(f"温度与价格相关系数: {corrs['temperature']['corr']:.4f} (p值: {corrs['temperature']['p_value']:.4f})")
        print(f"降水量与价格相关系数: {corrs['precipitation']['corr']:.4f} (p值: {corrs['precipitation']['p_value']:.4f})")
        print(f"湿度与价格相关系数: {corrs['humidity']['corr']:.4f} (p值: {corrs['humidity']['p_value']:.4f})")
    
    # 可视化
    plt.figure(figsize=(15, 10))
    
    # 为每个地区创建子图
    locations = df['location'].unique()
    n_locations = len(locations)
    rows = (n_locations + 1) // 2  # 计算需要的行数
    
    for i, location in enumerate(locations):
        loc_df = df[df['location'] == location]
        
        # 温度与价格
        plt.subplot(rows, 2, i+1)
        plt.scatter(loc_df['avg_temp'], loc_df['avg_price'], alpha=0.6)
        
        # 添加趋势线
        z = np.polyfit(loc_df['avg_temp'], loc_df['avg_price'], 1)
        p = np.poly1d(z)
        plt.plot(loc_df['avg_temp'], p(loc_df['avg_temp']), "r--")
        
        plt.title(f"{location} - 温度与价格关系")
        plt.xlabel("平均温度 (°C)")
        plt.ylabel("平均价格")
        plt.grid(True)
        
        # 添加相关系数文本
        corr = correlations[location]['temperature']['corr']
        plt.annotate(f"相关系数: {corr:.4f}", 
                     xy=(0.05, 0.95), 
                     xycoords='axes fraction',
                     bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))
    
    plt.tight_layout()
    plt.savefig(f"{herb_name}_{specification}_weather_correlation.png")
    plt.show()
    
    # 热力图显示所有相关性
    plt.figure(figsize=(12, 8))
    
    # 准备热力图数据
    heatmap_data = []
    for location in correlations:
        heatmap_data.append({
            'Location': location,
            'Temperature': correlations[location]['temperature']['corr'],
            'Precipitation': correlations[location]['precipitation']['corr'],
            'Humidity': correlations[location]['humidity']['corr']
        })
    
    heatmap_df = pd.DataFrame(heatmap_data)
    heatmap_df = heatmap_df.set_index('Location')
    
    # 绘制热力图
    sns.heatmap(heatmap_df, annot=True, cmap="coolwarm", center=0, fmt=".2f")
    plt.title(f"{herb_name} {specification} - 天气因素与价格相关性热力图")
    plt.tight_layout()
    plt.savefig(f"{herb_name}_{specification}_correlation_heatmap.png")
    plt.show()
    
    return df, correlations

# 示例使用
if __name__ == "__main__":
    analyze_weather_price_correlation("当归", "统")

3. 新闻情感分析与价格关联

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import psycopg2
from datetime import datetime, timedelta
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
import jieba
import re
from snownlp import SnowNLP

def connect_to_db():
    """连接到数据库"""
    conn = psycopg2.connect(
        host="localhost",
        database="herb_price_db",
        user="postgres",
        password="your_password"
    )
    return conn

def analyze_news_sentiment(herb_name, days=90):
    """分析与特定药材相关的新闻情感"""
    conn = connect_to_db()
    
    # 计算日期范围
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)
    
    # 查询新闻数据
    query = """
    SELECT 
        title,
        content,
        published_at,
        source
    FROM news
    WHERE published_at >= %s
      AND (title LIKE %s OR content LIKE %s)
    ORDER BY published_at
    """
    
    search_term = f"%{herb_name}%"
    df = pd.read_sql(query, conn, params=(start_date, search_term, search_term))
    
    # 查询同期价格数据
    price_query = """
    SELECT 
        recorded_at::date as date, 
        AVG(price::numeric) as avg_price
    FROM herb_prices
    WHERE herb_name = %s 
      AND recorded_at >= %s
    GROUP BY recorded_at::date
    ORDER BY recorded_at::date
    """
    
    price_df = pd.read_sql(price_query, conn, params=(herb_name, start_date))
    
    # 关闭数据库连接
    conn.close()
    
    if df.empty:
        print(f"没有找到与{herb_name}相关的新闻")
        return
    
    # 数据处理
    # 1. 情感分析
    df['sentiment'] = df['content'].apply(lambda x: SnowNLP(x).sentiments)
    
    # 2. 按日期聚合新闻情感
    df['published_at'] = pd.to_datetime(df['published_at']).dt.date
    sentiment_by_date = df.groupby('published_at')['sentiment'].mean().reset_index()
    sentiment_by_date['published_at'] = pd.to_datetime(sentiment_by_date['published_at'])
    
    # 3. 合并价格和情感数据
    price_df['date'] = pd.to_datetime(price_df['date'])
    merged_df = pd.merge(price_df, sentiment_by_date, 
                         left_on='date', right_on='published_at', 
                         how='outer')
    
    # 填充缺失值
    merged_df = merged_df.sort_values('date')
    merged_df['sentiment'] = merged_df['sentiment'].fillna(method='ffill')
    
    # 4. 计算情感与价格的相关性
    correlation = merged_df['avg_price'].corr(merged_df['sentiment'])
    
    # 5. 关键词提取
    def extract_keywords(text, top_n=10):
        # 清洗文本
        text = re.sub(r'[^\w\s]', '', text)
        # 分词
        words = jieba.cut(text)
        # 过滤停用词
        stopwords = set(['的', '了', '和', '是', '在', '有', '中', '与'])
        filtered_words = [word for word in words if word not in stopwords and len(word) > 1]
        # 统计词频
        word_counts = {}
        for word in filtered_words:
            word_counts[word] = word_counts.get(word, 0) + 1
        # 返回前N个高频词
        return sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:top_n]
    
    # 合并所有内容
    all_content = ' '.join(df['content'])
    keywords = extract_keywords(all_content)
    
    # 输出分析结果
    print(f"\n{herb_name} 相关新闻情感分析:")
    print(f"分析周期: {start_date.date()} 至 {end_date.date()}")
    print(f"共找到 {len(df)} 条相关新闻")
    
    print("\n情感分析:")
    print(f"平均情感得分: {df['sentiment'].mean():.4f} (0-1, 越接近1表示越正面)")
    print(f"情感与价格相关系数: {correlation:.4f}")
    
    print("\n热门关键词:")
    for word, count in keywords:
        print(f"{word}: {count}次")
    
    # 可视化
    plt.figure(figsize=(12, 10))
    
    # 绘制情感趋势
    plt.subplot(2, 1, 1)
    plt.plot(sentiment_by_date['published_at'], sentiment_by_date['sentiment'], 'b-', label='情感得分')
    plt.title(f"{herb_name} 相关新闻情感趋势")
    plt.xlabel("日期")
    plt.ylabel("情感得分 (0-1)")
    plt.legend()
    plt.grid(True)
    
    # 绘制情感与价格对比
    plt.subplot(2, 1, 2)
    
    # 创建两个Y轴
    fig, ax1 = plt.subplots()
    ax2 = ax1.twinx()
    
    # 绘制价格
    ax1.plot(merged_df['date'], merged_df['avg_price'], 'g-', label='价格')
    ax1.set_xlabel('日期')
    ax1.set_ylabel('价格', color='g')
    ax1.tick_params(axis='y', labelcolor='g')
    
    # 绘制情感
    ax2.plot(merged_df['date'], merged_df['sentiment'], 'b-', label='情感得分')
    ax2.set_ylabel('情感得分', color='b')
    ax2.tick_params(axis='y', labelcolor='b')
    
    plt.title(f"{herb_name} 价格与新闻情感对比")
    
    # 添加图例
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='best')
    
    plt.tight_layout()
    plt.savefig(f"{herb_name}_news_sentiment_analysis.png")
    plt.show()
    
    return df, merged_df

# 示例使用
if __name__ == "__main__":
    analyze_news_sentiment("当归")

4. 价格预测模型

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import psycopg2
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

def connect_to_db():
    """连接到数据库"""
    conn = psycopg2.connect(
        host="localhost",
        database="herb_price_db",
        user="postgres",
        password="your_password"
    )
    return conn

def create_sequences(data, seq_length):
    """创建时间序列数据"""
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        x = data[i:i+seq_length]
        y = data[i+seq_length]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

def predict_price(herb_name, specification, days_to_predict=30, history_days=365):
    """预测未来价格趋势"""
    conn = connect_to_db()
    
    # 计算日期范围
    end_date = datetime.now()
    start_date = end_date - timedelta(days=history_days)
    
    # 查询价格数据
    query = """
    SELECT 
        recorded_at::date as date, 
        AVG(price::numeric) as avg_price
    FROM herb_prices
    WHERE herb_name = %s 
      AND specification = %s
      AND recorded_at >= %s
    GROUP BY recorded_at::date
    ORDER BY recorded_at::date
    """
    
    df = pd.read_sql(query, conn, params=(herb_name, specification, start_date))
    
    # 查询天气数据
    weather_query = """
    SELECT 
        date,
        AVG(temperature) as avg_temp,
        AVG(precipitation) as avg_precip,
        AVG(humidity) as avg_humidity
    FROM weather_data
    WHERE date >= %s
    GROUP BY date
    ORDER BY date
    """
    
    weather_df = pd.read_sql(weather_query, conn, params=(start_date,))
    
    # 关闭数据库连接
    conn.close()
    
    if df.empty:
        print(f"没有找到{herb_name} {specification}的价格数据")
        return
    
    # 数据预处理
    df['date'] = pd.to_datetime(df['date'])
    weather_df['date'] = pd.to_datetime(weather_df['date'])
    
    # 合并价格和天气数据
    merged_df = pd.merge(df, weather_df, on='date', how='left')
    
    # 填充缺失值
    merged_df = merged_df.fillna(method='ffill')
    
    # 特征工程
    # 添加时间特征
    merged_df['month'] = merged_df['date'].dt.month
    merged_df['day_of_week'] = merged_df['date'].dt.dayofweek
    
    # 添加滞后特征
    for lag in [1, 3, 7, 14]:
        merged_df[f'price_lag_{lag}'] = merged_df['avg_price'].shift(lag)
    
    # 添加移动平均
    for window in [7, 14, 30]:
        merged_df[f'price_ma_{window}'] = merged_df['avg_price'].rolling(window=window).mean()
    
    # 添加价格变化率
    merged_df['price_pct_change'] = merged_df['avg_price'].pct_change()
    
    # 删除缺失值
    merged_df = merged_df.dropna()
    
    # 准备特征和目标变量
    features = ['avg_price', 'avg_temp', 'avg_precip', 'avg_humidity', 
                'month', 'day_of_week', 
                'price_lag_1', 'price_lag_3', 'price_lag_7', 'price_lag_14',
                'price_ma_7', 'price_ma_14', 'price_ma_30', 'price_pct_change']
    
    X = merged_df[features].values
    y = merged_df['avg_price'].values
    
    # 数据标准化
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    
    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()
    
    # 创建时间序列数据
    seq_length = 30  # 使用30天的数据预测下一天
    X_seq, y_seq = create_sequences(y_scaled, seq_length)
    
    # 重塑数据为LSTM输入格式 [samples, time steps, features]
    X_seq = X_seq.reshape(X_seq.shape[0], X_seq.shape[1], 1)
    
    # 分割训练集和测试集
    X_train, X_test, y_train, y_test = train_test_split(X_seq, y_seq, test_size=0.2, shuffle=False)
    
    # 构建LSTM模型
    model = Sequential([
        LSTM(50, return_sequences=True, input_shape=(seq_length, 1)),
        Dropout(0.2),
        LSTM(50),
        Dropout(0.2),
        Dense(1)
    ])
    
    model.compile(optimizer='adam', loss='mse')
    
    # 训练模型
    history = model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=32,
        validation_data=(X_test, y_test),
        verbose=1
    )
    
    # 评估模型
    y_pred_scaled = model.predict(X_test)
    y_pred = scaler_y.inverse_transform(y_pred_scaled)
    y_test_actual = scaler_y.inverse_transform(y_test.reshape(-1, 1))
    
    mse = mean_squared_error(y_test_actual, y_pred)
    mae = mean_absolute_error(y_test_actual, y_pred)
    r2 = r2_score(y_test_actual, y_pred)
    
    print(f"\n{herb_name} {specification} 价格预测模型评估:")
    print(f"均方误差 (MSE): {mse:.4f}")
    print(f"平均绝对误差 (MAE): {mae:.4f}")
    print(f"决定系数 (R²): {r2:.4f}")
    
    # 预测未来价格
    last_sequence = y_scaled[-seq_length:].reshape(1, seq_length, 1)
    
    future_predictions = []
    current_sequence = last_sequence.copy()
    
    for _ in range(days_to_predict):
        # 预测下一个值
        next_pred = model.predict(current_sequence)[0][0]
        future_predictions.append(next_pred)
        
        # 更新序列
        current_sequence = np.append(current_sequence[:, 1:, :], 
                                     [[next_pred]], 
                                     axis=1)
    
    # 转换回原始比例
    future_pred_actual = scaler_y.inverse_transform(np.array(future_predictions).reshape(-1, 1))
    
    # 创建未来日期
    last_date = merged_df['date'].iloc[-1]
    future_dates = [last_date + timedelta(days=i+1) for i in range(days_to_predict)]
    
    # 可视化
    plt.figure(figsize=(12, 8))
    
    # 绘制历史价格
    plt.plot(merged_df['date'], merged_df['avg_price'], 'b-', label='历史价格')
    
    # 绘制预测价格
    plt.plot(future_dates, future_pred_actual, 'r--', label='预测价格')
    
    # 添加置信区间 (简单估计)
    std_dev = np.std(merged_df['avg_price'])
    plt.fill_between(future_dates, 
                     future_pred_actual.flatten() - 1.96 * std_dev, 
                     future_pred_actual.flatten() + 1.96 * std_dev, 
                     color='r', alpha=0.2)
    
    plt.title(f"{herb_name} {specification} 价格预测 (未来{days_to_predict}天)")
    plt.xlabel("日期")
    plt.ylabel("价格")
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(f"{herb_name}_{specification}_price_prediction.png")
    plt.show()
    
    # 返回预测结果
    prediction_df = pd.DataFrame({
        'date': future_dates,
        'predicted_price': future_pred_actual.flatten()
    })
    
    return prediction_df, model

# 示例使用
if __name__ == "__main__":
    predict_price("当归", "统")

5. 市场比较分析

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import psycopg2
import seaborn as sns
from datetime import datetime, timedelta
import numpy as np

def connect_to_db():
    """连接到数据库"""
    conn = psycopg2.connect(
        host="localhost",
        database="herb_price_db",
        user="postgres",
        password="your_password"
    )
    return conn

def analyze_market_comparison(herb_name, specification, days=90):
    """分析不同市场的价格比较"""
    conn = connect_to_db()
    
    # 计算日期范围
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)
    
    # 查询价格数据
    query = """
    SELECT 
        location,
        recorded_at::date as date, 
        AVG(price::numeric) as avg_price,
        MIN(price::numeric) as min_price,
        MAX(price::numeric) as max_price,
        source
    FROM herb_prices
    WHERE herb_name = %s 
      AND specification = %s
      AND recorded_at >= %s
    GROUP BY location, recorded_at::date, source
    ORDER BY location, recorded_at::date
    """
    
    df = pd.read_sql(query, conn, params=(herb_name, specification, start_date))
    
    # 关闭数据库连接
    conn.close()
    
    if df.empty:
        print(f"没有找到{herb_name} {specification}的价格数据")
        return
    
    # 数据分析
    # 1. 计算各市场平均价格
    market_avg = df.groupby('location')['avg_price'].mean().sort_values(ascending=False)
    
    # 2. 计算各市场价格波动性
    market_volatility = df.groupby('location')['avg_price'].std()
    
    # 3. 计算各市场价格范围
    market_range = df.groupby('location').apply(lambda x: x['max_price'].max() - x['min_price'].min())
    
    # 4. 计算各市场最新价格
    df['date'] = pd.to_datetime(df['date'])
    latest_date = df['date'].max()
    latest_prices = df[df['date'] == latest_date].groupby('location')['avg_price'].mean()
    
    # 5. 计算各市场价格趋势
    trends = {}
    for location in df['location'].unique():
        loc_df = df[df['location'] == location].sort_values('date')
        if len(loc_df) > 1:
            X = np.array(range(len(loc_df))).reshape(-1, 1)
            y = loc_df['avg_price'].values
            from sklearn.linear_model import LinearRegression
            model = LinearRegression()
            model.fit(X, y)
            trends[location] = model.coef_[0]
    
    # 输出分析结果
    print(f"\n{herb_name} {specification} 市场价格比较:")
    print(f"分析周期: {start_date.date()} 至 {end_date.date()}")
    
    print("\n各市场平均价格:")
    for location, avg_price in market_avg.items():
        print(f"{location}: {avg_price:.2f}元")
    
    print("\n各市场价格波动性 (标准差):")
    for location, std in market_volatility.items():
        print(f"{location}: {std:.2f}")
    
    print("\n各市场价格趋势:")
    for location, slope in trends.items():
        trend_direction = "上升" if slope > 0 else "下降"
        print(f"{location}: {trend_direction}, 斜率={slope:.4f}")
    
    print("\n各市场最新价格 ({})".format(latest_date.strftime('%Y-%m-%d')))
    for location, price in latest_prices.items():
        print(f"{location}: {price:.2f}元")
    
    # 可视化
    plt.figure(figsize=(15, 10))
    
    # 1. 各市场平均价格条形图
    plt.subplot(2, 2, 1)
    market_avg.plot(kind='bar', color='skyblue')
    plt.title('各市场平均价格')
    plt.xlabel('市场')
    plt.ylabel('平均价格 (元)')
    plt.xticks(rotation=45)
    plt.grid(axis='y')
    
    # 2. 各市场价格波动性条形图
    plt.subplot(2, 2, 2)
    market_volatility.plot(kind='bar', color='salmon')
    plt.title('各市场价格波动性')
    plt.xlabel('市场')
    plt.ylabel('标准差')
    plt.xticks(rotation=45)
    plt.grid(axis='y')
    
    # 3. 各市场价格范围条形图
    plt.subplot(2, 2, 3)
    market_range.plot(kind='bar', color='lightgreen')
    plt.title('各市场价格范围')
    plt.xlabel('市场')
    plt.ylabel('价格范围 (元)')
    plt.xticks(rotation=45)
    plt.grid(axis='y')
    
    # 4. 各市场价格趋势
    plt.subplot(2, 2, 4)
    
    # 为每个市场绘制价格趋势线
    for location in df['location'].unique():
        loc_df = df[df['location'] == location].sort_values('date')
        plt.plot(loc_df['date'], loc_df['avg_price'], label=location)
    
    plt.title('各市场价格趋势')
    plt.xlabel('日期')
    plt.ylabel('价格 (元)')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(f"{herb_name}_{specification}_market_comparison.png")
    plt.show()
    
    # 5. 热力图 - 各市场价格相关性
    plt.figure(figsize=(10, 8))
    
    # 创建透视表
    pivot_df = df.pivot_table(index='date', columns='location', values='avg_price')
    
    # 计算相关性
    corr_matrix = pivot_df.corr()
    
    # 绘制热力图
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
    plt.title(f"{herb_name} {specification} - 各市场价格相关性")
    plt.tight_layout()
    plt.savefig(f"{herb_name}_{specification}_market_correlation.png")
    plt.show()
    
    return df

# 示例使用
if __name__ == "__main__":
    analyze_market_comparison("当归", "统")

6. 季节性分析

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import psycopg2
import seaborn as sns
from datetime import datetime, timedelta
import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose

def connect_to_db():
    """连接到数据库"""
    conn = psycopg2.connect(
        host="localhost",
        database="herb_price_db",
        user="postgres",
        password="your_password"
    )
    return conn

def analyze_seasonality(herb_name, specification, years=3):
    """分析药材价格的季节性变化"""
    conn = connect_to_db()
    
    # 计算日期范围
    end_date = datetime.now()
    start_date = end_date - timedelta(days=365 * years)
    
    # 查询价格数据
    query = """
    SELECT 
        recorded_at::date as date, 
        AVG(price::numeric) as avg_price
    FROM herb_prices
    WHERE herb_name = %s 
      AND specification = %s
      AND recorded_at >= %s
    GROUP BY recorded_at::date
    ORDER BY recorded_at::date
    """
    
    df = pd.read_sql(query, conn, params=(herb_name, specification, start_date))
    
    # 关闭数据库连接
    conn.close()
    
    if df.empty:
        print(f"没有找到{herb_name} {specification}的价格数据")
        return
    
    # 数据预处理
    df['date'] = pd.to_datetime(df['date'])
    df.set_index('date', inplace=True)
    
    # 确保数据是连续的日期序列
    df = df.asfreq('D')
    
    # 填充缺失值
    df['avg_price'] = df['avg_price'].interpolate(method='time')
    
    # 1. 按月份分析
    df['month'] = df.index.month
    df['year'] = df.index.year
    
    monthly_avg = df.groupby('month')['avg_price'].mean()
    monthly_std = df.groupby('month')['avg_price'].std()
    
    # 2. 按季度分析
    df['quarter'] = df.index.quarter
    quarterly_avg = df.groupby('quarter')['avg_price'].mean()
    
    # 3. 按年度分析
    yearly_avg = df.groupby('year')['avg_price'].mean()
    
    # 4. 时间序列分解
    # 重采样为月度数据以便分解
    monthly_data = df['avg_price'].resample('M').mean()
    
    # 确保有足够的数据点
    if len(monthly_data) >= 24:  # 至少需要2年的数据
        result = seasonal_decompose(monthly_data, model='additive', period=12)
        trend = result.trend
        seasonal = result.seasonal
        residual = result.resid
    else:
        print("数据点不足，无法进行时间序列分解")
        trend = seasonal = residual = None
    
    # 输出分析结果
    print(f"\n{herb_name} {specification} 季节性分析:")
    print(f"分析周期: {start_date.date()} 至 {end_date.date()}")
    
    print("\n月度平均价格:")
    month_names = ['一月', '二月', '三月', '四月', '五月', '六月', 
                   '七月', '八月', '九月', '十月', '十一月', '十二月']
    for i, avg_price in enumerate(monthly_avg, 1):
        print(f"{month_names[i-1]}: {avg_price:.2f}元")
    
    print("\n季度平均价格:")
    quarter_names = ['第一季度', '第二季度', '第三季度', '第四季度']
    for i, avg_price in enumerate(quarterly_avg, 1):
        print(f"{quarter_names[i-1]}: {avg_price:.2f}元")
    
    print("\n年度平均价格:")
    for year, avg_price in yearly_avg.items():
        print(f"{year}年: {avg_price:.2f}元")
    
    # 可视化
    plt.figure(figsize=(15, 12))
    
    # 1. 月度平均价格
    plt.subplot(2, 2, 1)
    monthly_avg.plot(kind='bar', yerr=monthly_std, capsize=4, color='skyblue')
    plt.title('月度平均价格')
    plt.xlabel('月份')
    plt.ylabel('平均价格 (元)')
    plt.xticks(range(12), month_names, rotation=45)
    plt.grid(axis='y')
    
    # 2. 季度平均价格
    plt.subplot(2, 2, 2)
    quarterly_avg.plot(kind='bar', color='salmon')
    plt.title('季度平均价格')
    plt.xlabel('季度')
    plt.ylabel('平均价格 (元)')
    plt.xticks(range(4), quarter_names, rotation=45)
    plt.grid(axis='y')
    
    # 3. 年度平均价格
    plt.subplot(2, 2, 3)
    yearly_avg.plot(kind='bar', color='lightgreen')
    plt.title('年度平均价格')
    plt.xlabel('年份')
    plt.ylabel('平均价格 (元)')
    plt.grid(axis='y')
    
    # 4. 月度热图
    plt.subplot(2, 2, 4)
    
    # 创建月度价格透视表
    pivot_df = df.pivot_table(index=df.index.month, columns=df.index.year, values='avg_price')
    
    # 绘制热图
    sns.heatmap(pivot_df, annot=True, fmt='.1f', cmap='YlGnBu')
    plt.title('月度价格热图')
    plt.xlabel('年份')
    plt.ylabel('月份')
    plt.yticks(range(1, 13), month_names)
    
    plt.tight_layout()
    plt.savefig(f"{herb_name}_{specification}_seasonality_analysis.png")
    plt.show()
    
    # 如果有时间序列分解结果，绘制分解图
    if trend is not None:
        plt.figure(figsize=(12, 10))
        
        plt.subplot(4, 1, 1)
        plt.plot(monthly_data)
        plt.title('原始时间序列')
        plt.grid(True)
        
        plt.subplot(4, 1, 2)
        plt.plot(trend)
        plt.title('趋势')
        plt.grid(True)
        
        plt.subplot(4, 1, 3)
        plt.plot(seasonal)
        plt.title('季节性')
        plt.grid(True)
        
        plt.subplot(4, 1, 4)
        plt.plot(residual)
        plt.title('残差')
        plt.grid(True)
        
        plt.tight_layout()
        plt.savefig(f"{herb_name}_{specification}_time_series_decomposition.png")
        plt.show()
    
    return df

# 示例使用
if __name__ == "__main__":
    analyze_seasonality("当归", "统")

7. 综合分析报告生成器

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import psycopg2
from datetime import datetime, timedelta
import numpy as np
import os
from jinja2 import Template
import webbrowser

def connect_to_db():
    """连接到数据库"""
    conn = psycopg2.connect(
        host="localhost",
        database="herb_price_db",
        user="postgres",
        password="your_password"
    )
    return conn

def generate_report(herb_name, specification, days=90):
    """生成综合分析报告"""
    conn = connect_to_db()
    
    # 计算日期范围
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)
    
    # 查询价格数据
    price_query = """
    SELECT 
        recorded_at::date as date, 
        AVG(price::numeric) as avg_price,
        MIN(price::numeric) as min_price,
        MAX(price::numeric) as max_price,
        location,
        source
    FROM herb_prices
    WHERE herb_name = %s 
      AND specification = %s
      AND recorded_at >= %s
    GROUP BY recorded_at::date, location, source
    ORDER BY recorded_at::date
    """
    
    price_df = pd.read_sql(price_query, conn, params=(herb_name, specification, start_date))
    
    # 查询天气数据
    weather_query = """
    SELECT 
        date,
        location,
        AVG(temperature) as avg_temp,
        AVG(precipitation) as avg_precip,
        AVG(humidity) as avg_humidity
    FROM weather_data
    WHERE date >= %s
    GROUP BY date, location
    ORDER BY date, location
    """
    
    weather_df = pd.read_sql(weather_query, conn, params=(start_date,))
    
    # 查询新闻数据
    news_query = """
    SELECT 
        title,
        content,
        published_at,
        source,
        url
    FROM news
    WHERE published_at >= %s
      AND (title LIKE %s OR content LIKE %s)
    ORDER BY published_at DESC
    LIMIT 10
    """
    
    search_term = f"%{herb_name}%"
    news_df = pd.read_sql(news_query, conn, params=(start_date, search_term, search_term))
    
    # 关闭数据库连接
    conn.close()
    
    if price_df.empty:
        print(f"没有找到{herb_name} {specification}的价格数据")
        return
    
    # 数据处理
    price_df['date'] = pd.to_datetime(price_df['date'])
    
    # 1. 价格趋势分析
    overall_avg = price_df.groupby('date')['avg_price'].mean().reset_index()
    
    # 计算价格变化率
    overall_avg['price_change'] = overall_avg['avg_price'].pct_change() * 100
    
    # 计算价格趋势 (线性回归)
    from sklearn.linear_model import LinearRegression
    
    X = np.array(range(len(overall_avg))).reshape(-1, 1)
    y = overall_avg['avg_price'].values
    model = LinearRegression()
    model.fit(X, y)
    trend_slope = model.coef_[0]
    trend_direction = "上升" if trend_slope > 0 else "下降"
    
    # 2. 市场比较分析
    market_avg = price_df.groupby('location')['avg_price'].mean().sort_values(ascending=False)
    market_volatility = price_df.groupby('location')['avg_price'].std()
    
    # 3. 天气相关性分析
    # 合并价格和天气数据
    weather_df['date'] = pd.to_datetime(weather_df['date'])
    
    # 按地区合并
    weather_correlations = {}
    
    for location in price_df['location'].unique():
        loc_price = price_df[price_df['location'] == location].groupby('date')['avg_price'].mean().reset_index()
        loc_weather = weather_df[weather_df['location'] == location]
        
        if not loc_weather.empty and not loc_price.empty:
            merged = pd.merge(loc_price, loc_weather, on='date', how='inner')
            
            if len(merged) > 5:  # 确保有足够的数据点
                from scipy.stats import pearsonr
                
                temp_corr, temp_p = pearsonr(merged['avg_price'], merged['avg_temp'])
                precip_corr, precip_p = pearsonr(merged['avg_price'], merged['avg_precip'])
                
                weather_correlations[location] = {
                    'temperature': {'corr': temp_corr, 'p_value': temp_p},
                    'precipitation': {'corr': precip_corr, 'p_value': precip_p}
                }
    
    # 4. 季节性分析
    # 添加月份
    price_df['month'] = price_df['date'].dt.month
    monthly_avg = price_df.groupby('month')['avg_price'].mean()
    
    # 5. 价格预测
    # 简单的移动平均预测
    last_price = overall_avg['avg_price'].iloc[-1]
    avg_change = overall_avg['price_change'].mean()
    predicted_price = last_price * (1 + avg_change/100)
    
    # 生成报告数据
    report_data = {
        'herb_name': herb_name,
        'specification': specification,
        'analysis_period': {
            'start': start_date.strftime('%Y-%m-%d'),
            'end': end_date.strftime('%Y-%m-%d')
        },
        'price_summary': {
            'current_price': overall_avg['avg_price'].iloc[-1],
            'avg_price': overall_avg['avg_price'].mean(),
            'min_price': price_df['min_price'].min(),
            'max_price': price_df['max_price'].max(),
            'price_change': overall_avg['price_change'].iloc[-1] if len(overall_avg) > 1 else 0,
            'trend': {
                'direction': trend_direction,
                'slope': trend_slope
            }
        },
        'market_comparison': {
            'markets': [{'name': loc, 'avg_price': price, 'volatility': market_volatility[loc]} 
                       for loc, price in market_avg.items()]
        },
        'weather_correlation': {
            'correlations': [{'location': loc, 
                             'temperature_corr': data['temperature']['corr'],
                             'precipitation_corr': data['precipitation']['corr']} 
                            for loc, data in weather_correlations.items()]
        },
        'seasonality': {
            'monthly_avg': [{'month': i, 'avg_price': monthly_avg[i]} for i in range(1, 13) if i in monthly_avg]
        },
        'prediction': {
            'next_price': predicted_price
        },
        'news': {
            'articles': [{'title': row['title'], 
                         'date': row['published_at'], 
                         'source': row['source'],
                         'url': row['url']} 
                        for _, row in news_df.iterrows()]
        },
        'generated_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    
    # 生成HTML报告
    html_template = """
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <title>{{ herb_name }} {{ specification }} 市场分析报告</title>
        <style>
            body {
                font-family: Arial, sans-serif;
                line-height: 1.6;
                margin: 0;
                padding: 20px;
                color: #333;
            }
            .container {
                max-width: 1200px;
                margin: 0 auto;
            }
            h1, h2, h3 {
                color: #2c3e50;
            }
            .header {
                text-align: center;
                margin-bottom: 30px;
                padding-bottom: 20px;
                border-bottom: 1px solid #eee;
            }
            .section {
                margin-bottom: 30px;
                padding: 20px;
                background-color: #f9f9f9;
                border-radius: 5px;
            }
            .summary-box {
                display: flex;
                flex-wrap: wrap;
                justify-content: space-between;
            }
            .summary-item {
                flex: 0 0 30%;
                margin-bottom: 15px;
                padding: 15px;
                background-color: #fff;
                border-radius: 5px;
                box-shadow: 0 2px 5px rgba(0,0,0,0.1);
            }
            .summary-item h4 {
                margin-top: 0;
                color: #7f8c8d;
            }
            .summary-item p {
                font-size: 24px;
                font-weight: bold;
                margin: 5px 0;
            }
            .summary-item .change {
                font-size: 14px;
            }
            .change.positive {
                color: #27ae60;
            }
            .change.negative {
                color: #e74c3c;
            }
            table {
                width: 100%;
                border-collapse: collapse;
                margin-bottom: 20px;
            }
            th, td {
                padding: 12px 15px;
                text-align: left;
                border-bottom: 1px solid #ddd;
            }
            th {
                background-color: #f2f2f2;
            }
            tr:hover {
                background-color: #f5f5f5;
            }
            .footer {
                text-align: center;
                margin-top: 50px;
                padding-top: 20px;
                border-top: 1px solid #eee;
                color: #7f8c8d;
            }
        </style>
    </head>
    <body>
        <div class="container">
            <div class="header">
                <h1>{{ herb_name }} {{ specification }} 市场分析报告</h1>
                <p>分析周期: {{ analysis_period.start }} 至 {{ analysis_period.end }}</p>
            </div>
            
            <div class="section">
                <h2>价格概览</h2>
                <div class="summary-box">
                    <div class="summary-item">
                        <h4>当前价格</h4>
                        <p>{{ price_summary.current_price|round(2) }} 元</p>
                        {% if price_summary.price_change > 0 %}
                            <span class="change positive">+{{ price_summary.price_change|round(2) }}%</span>
                        {% else %}
                            <span class="change negative">{{ price_summary.price_change|round(2) }}%</span>
                        {% endif %}
                    </div>
                    <div class="summary-item">
                        <h4>平均价格</h4>
                        <p>{{ price_summary.avg_price|round(2) }} 元</p>
                    </div>
                    <div class="summary-item">
                        <h4>价格区间</h4>
                        <p>{{ price_summary.min_price|round(2) }} - {{ price_summary.max_price|round(2) }} 元</p>
                    </div>
                    <div class="summary-item">
                        <h4>价格趋势</h4>
                        {% if price_summary.trend.direction == "上升" %}
                            <p class="change positive">{{ price_summary.trend.direction }}</p>
                        {% else %}
                            <p class="change negative">{{ price_summary.trend.direction }}</p>
                        {% endif %}
                    </div>
                    <div class="summary-item">
                        <h4>预测价格</h4>
                        <p>{{ prediction.next_price|round(2) }} 元</p>
                    </div>
                </div>
            </div>
            
            <div class="section">
                <h2>市场比较</h2>
                <table>
                    <thead>
                        <tr>
                            <th>市场</th>
                            <th>平均价格 (元)</th>
                            <th>价格波动性</th>
                        </tr>
                    </thead>
                    <tbody>
                        {% for market in market_comparison.markets %}
                        <tr>
                            <td>{{ market.name }}</td>
                            <td>{{ market.avg_price|round(2) }}</td>
                            <td>{{ market.volatility|round(2) }}</td>
                        </tr>
                        {% endfor %}
                    </tbody>
                </table>
            </div>
            
            <div class="section">
                <h2>天气相关性</h2>
                <table>
                    <thead>
                        <tr>
                            <th>地区</th>
                            <th>温度相关性</th>
                            <th>降水量相关性</th>
                        </tr>
                    </thead>
                    <tbody>
                        {% for corr in weather_correlation.correlations %}
                        <tr>
                            <td>{{ corr.location }}</td>
                            <td>{{ corr.temperature_corr|round(4) }}</td>
                            <td>{{ corr.precipitation_corr|round(4) }}</td>
                        </tr>
                        {% endfor %}
                    </tbody>
                </table>
                <p>注: 相关系数范围从-1到1，正值表示正相关，负值表示负相关，接近0表示相关性弱。</p>
            </div>
            
            <div class="section">
                <h2>季节性分析</h2>
                <table>
                    <thead>
                        <tr>
                            <th>月份</th>
                            <th>平均价格 (元)</th>
                        </tr>
                    </thead>
                    <tbody>
                        {% for month_data in seasonality.monthly_avg %}
                        <tr>
                            <td>{{ month_data.month }}月</td>
                            <td>{{ month_data.avg_price|round(2) }}</td>
                        </tr>
                        {% endfor %}
                    </tbody>
                </table>
            </div>
            
            <div class="section">
                <h2>相关新闻</h2>
                {% if news.articles %}
                <table>
                    <thead>
                        <tr>
                            <th>日期</th>
                            <th>标题</th>
                            <th>来源</th>
                        </tr>
                    </thead>
                    <tbody>
                        {% for article in news.articles %}
                        <tr>
                            <td>{{ article.date }}</td>
                            <td>
                                {% if article.url %}
                                <a href="{{ article.url }}" target="_blank">{{ article.title }}</a>
                                {% else %}
                                {{ article.title }}
                                {% endif %}
                            </td>
                            <td>{{ article.source }}</td>
                        </tr>
                        {% endfor %}
                    </tbody>
                </table>
                {% else %}
                <p>没有找到相关新闻</p>
                {% endif %}
            </div>
            
            <div class="footer">
                <p>报告生成时间: {{ generated_at }}</p>
                <p>© 2023 中药材价格分析系统</p>
            </div>
        </div>
    </body>
    </html>
    """
    
    # 使用Jinja2渲染模板
    template = Template(html_template)
    html_content = template.render(**report_data)
    
    # 保存HTML报告
    report_dir = os.path.join(os.path.dirname(os.path.abspath(__file__)), 'reports')
    os.makedirs(report_dir, exist_ok=True)
    
    report_filename = f"{herb_name}_{specification}_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.html"
    report_path = os.path.join(report_dir, report_filename)
    
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    print(f"报告已生成: {report_path}")
    
    # 自动打开报告
    webbrowser.open('file://' + os.path.abspath(report_path))
    
    return report_path

# 示例使用
if __name__ == "__main__":
    generate_report("当归", "统")

8. 数据可视化仪表板

In [ ]:
import dash
from dash import dcc, html, Input, Output, State
import plotly.graph_objs as go
import plotly.express as px
import pandas as pd
import psycopg2
from datetime import datetime, timedelta
import numpy as np
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr

# 初始化Dash应用
app = dash.Dash(__name__, title="中药材价格分析仪表板")

# 数据库连接函数
def connect_to_db():
    """连接到数据库"""
    conn = psycopg2.connect(
        host="localhost",
        database="herb_price_db",
        user="postgres",
        password="your_password"
    )
    return conn

# 获取药材列表
def get_herbs():
    conn = connect_to_db()
    cursor = conn.cursor()
    cursor.execute("SELECT DISTINCT herb_name FROM herb_prices ORDER BY herb_name")
    herbs = [row[0] for row in cursor.fetchall()]
    cursor.close()
    conn.close()
    return herbs

# 获取规格列表
def get_specifications(herb_name):
    conn = connect_to_db()
    cursor = conn.cursor()
    cursor.execute("SELECT DISTINCT specification FROM herb_prices WHERE herb_name = %s ORDER BY specification", (herb_name,))
    specs = [row[0] for row in cursor.fetchall()]
    cursor.close()
    conn.close()
    return specs

# 获取地区列表
def get_locations():
    conn = connect_to_db()
    cursor = conn.cursor()
    cursor.execute("SELECT DISTINCT location FROM herb_prices ORDER BY location")
    locations = [row[0] for row in cursor.fetchall()]
    cursor.close()
    conn.close()
    return locations

# 获取价格数据
def get_price_data(herb_name, specification, days=90):
    conn = connect_to_db()
    
    # 计算日期范围
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)
    
    # 查询价格数据
    query = """
    SELECT 
        recorded_at::date as date, 
        AVG(price::numeric) as avg_price,
        MIN(price::numeric) as min_price,
        MAX(price::numeric) as max_price,
        location
    FROM herb_prices
    WHERE herb_name = %s 
      AND specification = %s
      AND recorded_at >= %s
    GROUP BY recorded_at::date, location
    ORDER BY recorded_at::date
    """
    
    df = pd.read_sql(query, conn, params=(herb_name, specification, start_date))
    conn.close()
    
    return df

# 获取天气数据
def get_weather_data(location, days=90):
    conn = connect_to_db()
    
    # 计算日期范围
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)
    
    # 查询天气数据
    query = """
    SELECT 
        date,
        temperature as avg_temp,
        precipitation as avg_precip,
        humidity as avg_humidity
    FROM weather_data
    WHERE location = %s
      AND date >= %s
    ORDER BY date
    """
    
    df = pd.read_sql(query, conn, params=(location, start_date))
    conn.close()
    
    return df

# 应用布局
app.layout = html.Div([
    html.H1("中药材价格分析仪表板", style={'textAlign': 'center', 'marginBottom': 30}),
    
    # 过滤器部分
    html.Div([
        html.Div([
            html.Label("选择药材:"),
            dcc.Dropdown(
                id='herb-dropdown',
                options=[{'label': herb, 'value': herb} for herb in get_herbs()],
                value=get_herbs()[0] if get_herbs() else None
            )
        ], style={'width': '30%', 'display': 'inline-block', 'marginRight': '2%'}),
        
        html.Div([
            html.Label("选择规格:"),
            dcc.Dropdown(id='spec-dropdown')
        ], style={'width': '30%', 'display': 'inline-block', 'marginRight': '2%'}),
        
        html.Div([
            html.Label("时间范围:"),
            dcc.Dropdown(
                id='time-dropdown',
                options=[
                    {'label': '最近30天', 'value': 30},
                    {'label': '最近90天', 'value': 90},
                    {'label': '最近180天', 'value': 180},
                    {'label': '最近365天', 'value': 365}
                ],
                value=90
            )
        ], style={'width': '30%', 'display': 'inline-block'}),
        
        html.Button('查询', id='query-button', n_clicks=0, 
                   style={'marginTop': 20, 'padding': '10px 20px', 'backgroundColor': '#4CAF50', 'color': 'white', 'border': 'none'})
    ], style={'marginBottom': 30, 'padding': 20, 'backgroundColor': '#f9f9f9', 'borderRadius': 5}),
    
    # 价格趋势图
    html.Div([
        html.H2("价格趋势", style={'textAlign': 'center'}),
        dcc.Graph(id='price-trend-graph')
    ], style={'marginBottom': 30, 'padding': 20, 'backgroundColor': '#f9f9f9', 'borderRadius': 5}),
    
    # 市场比较和天气相关性
    html.Div([
        html.Div([
            html.H2("市场价格比较", style={'textAlign': 'center'}),
            dcc.Graph(id='market-comparison-graph')
        ], style={'width': '48%', 'display': 'inline-block'}),
        
        html.Div([
            html.H2("天气与价格相关性", style={'textAlign': 'center'}),
            dcc.Dropdown(
                id='location-dropdown',
                placeholder="选择地区查看天气相关性"
            ),
            dcc.Graph(id='weather-correlation-graph')
        ], style={'width': '48%', 'display': 'inline-block', 'float': 'right'})
    ], style={'marginBottom': 30}),
    
    # 季节性分析
    html.Div([
        html.H2("季节性分析", style={'textAlign': 'center'}),
        dcc.Graph(id='seasonality-graph')
    ], style={'marginBottom': 30, 'padding': 20, 'backgroundColor': '#f9f9f9', 'borderRadius': 5}),
    
    # 价格预测
    html.Div([
        html.H2("价格预测 (未来30天)", style={'textAlign': 'center'}),
        dcc.Graph(id='prediction-graph')
    ], style={'marginBottom': 30, 'padding': 20, 'backgroundColor': '#f9f9f9', 'borderRadius': 5}),
    
    # 页脚
    html.Div([
        html.P("© 2023 中药材价格分析系统", style={'textAlign': 'center'})
    ], style={'marginTop': 50, 'paddingTop': 20, 'borderTop': '1px solid #eee'})
])

# 更新规格下拉菜单
@app.callback(
    Output('spec-dropdown', 'options'),
    Output('spec-dropdown', 'value'),
    Input('herb-dropdown', 'value')
)
def update_spec_dropdown(herb_name):
    if not herb_name:
        return [], None
    
    specs = get_specifications(herb_name)
    options = [{'label': spec, 'value': spec} for spec in specs]
    value = specs[0] if specs else None
    
    return options, value

# 更新地区下拉菜单
@app.callback(
    Output('location-dropdown', 'options'),
    Output('location-dropdown', 'value'),
    Input('query-button', 'n_clicks'),
    State('herb-dropdown', 'value'),
    State('spec-dropdown', 'value'),
    State('time-dropdown', 'value')
)
def update_location_dropdown(n_clicks, herb_name, specification, days):
    if n_clicks == 0 or not herb_name or not specification:
        return [], None
    
    df = get_price_data(herb_name, specification, days)
    locations = df['location'].unique()
    
    options = [{'label': loc, 'value': loc} for loc in locations]
    value = locations[0] if len(locations) > 0 else None
    
    return options, value

# 更新价格趋势图
@app.callback(
    Output('price-trend-graph', 'figure'),
    Input('query-button', 'n_clicks'),
    State('herb-dropdown', 'value'),
    State('spec-dropdown', 'value'),
    State('time-dropdown', 'value')
)
def update_price_trend(n_clicks, herb_name, specification, days):
    if n_clicks == 0 or not herb_name or not specification:
        return {
            'data': [],
            'layout': {
                'title': '请选择药材和规格并点击查询',
                'xaxis': {'title': '日期'},
                'yaxis': {'title': '价格 (元)'}
            }
        }
    
    df = get_price_data(herb_name, specification, days)
    
    if df.empty:
        return {
            'data': [],
            'layout': {
                'title': f'没有找到{herb_name} {specification}的价格数据',
                'xaxis': {'title': '日期'},
                'yaxis': {'title': '价格 (元)'}
            }
        }
    
    # 按日期聚合
    df['date'] = pd.to_datetime(df['date'])
    overall_avg = df.groupby('date')[['avg_price', 'min_price', 'max_price']].mean().reset_index()
    
    # 创建图表
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=overall_avg['date'],
        y=overall_avg['avg_price'],
        mode='lines+markers',
        name='平均价格',
        line=dict(color='royalblue', width=3)
    ))
    
    fig.add_trace(go.Scatter(
        x=overall_avg['date'],
        y=overall_avg['min_price'],
        mode='lines',
        name='最低价格',
        line=dict(color='lightblue', width=2, dash='dash')
    ))
    
    fig.add_trace(go.Scatter(
        x=overall_avg['date'],
        y=overall_avg['max_price'],
        mode='lines',
        name='最高价格',
        line=dict(color='darkblue', width=2, dash='dash')
    ))
    
    # 添加趋势线
    X = np.array(range(len(overall_avg))).reshape(-1, 1)
    y = overall_avg['avg_price'].values
    model = LinearRegression()
    model.fit(X, y)
    trend = model.predict(X)
    
    fig.add_trace(go.Scatter(
        x=overall_avg['date'],
        y=trend,
        mode='lines',
        name='趋势线',
        line=dict(color='red', width=2, dash='dot')
    ))
    
    fig.update_layout(
        title=f'{herb_name} {specification} 价格趋势',
        xaxis_title='日期',
        yaxis_title='价格 (元)',
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        ),
        margin=dict(l=40, r=40, t=60, b=40)
    )
    
    return fig

# 更新市场比较图
@app.callback(
    Output('market-comparison-graph', 'figure'),
    Input('query-button', 'n_clicks'),
    State('herb-dropdown', 'value'),
    State('spec-dropdown', 'value'),
    State('time-dropdown', 'value')
)
def update_market_comparison(n_clicks, herb_name, specification, days):
    if n_clicks == 0 or not herb_name or not specification:
        return {
            'data': [],
            'layout': {
                'title': '请选择药材和规格并点击查询',
                'xaxis': {'title': '市场'},
                'yaxis': {'title': '平均价格 (元)'}
            }
        }
    
    df = get_price_data(herb_name, specification, days)
    
    if df.empty:
        return {
            'data': [],
            'layout': {
                'title': f'没有找到{herb_name} {specification}的价格数据',
                'xaxis': {'title': '市场'},
                'yaxis': {'title': '平均价格 (元)'}
            }
        }
    
    # 按市场聚合
    market_avg = df.groupby('location')['avg_price'].mean().reset_index()
    market_avg = market_avg.sort_values('avg_price', ascending=False)
    
    # 创建图表
    fig = px.bar(
        market_avg, 
        x='location', 
        y='avg_price',
        labels={'location': '市场', 'avg_price': '平均价格 (元)'},
        title=f'{herb_name} {specification} 市场价格比较',
        color='avg_price',
        color_continuous_scale='Viridis'
    )
    
    fig.update_layout(
        xaxis_tickangle=-45,
        margin=dict(l=40, r=40, t=60, b=80)
    )
    
    return fig

# 更新天气相关性图
@app.callback(
    Output('weather-correlation-graph', 'figure'),
    Input('location-dropdown', 'value'),
    State('herb-dropdown', 'value'),
    State('spec-dropdown', 'value'),
    State('time-dropdown', 'value')
)
def update_weather_correlation(location, herb_name, specification, days):
    if not location or not herb_name or not specification:
        return {
            'data': [],
            'layout': {
                'title': '请选择地区查看天气相关性',
                'xaxis': {'title': '日期'},
                'yaxis': {'title': '数值'}
            }
        }
    
    # 获取价格数据
    price_df = get_price_data(herb_name, specification, days)
    price_df = price_df[price_df['location'] == location]
    
    if price_df.empty:
        return {
            'data': [],
            'layout': {
                'title': f'没有找到{location}的价格数据',
                'xaxis': {'title': '日期'},
                'yaxis': {'title': '数值'}
            }
        }
    
    # 获取天气数据
    weather_df = get_weather_data(location, days)
    
    if weather_df.empty:
        return {
            'data': [],
            'layout': {
                'title': f'没有找到{location}的天气数据',
                'xaxis': {'title': '日期'},
                'yaxis': {'title': '数值'}
            }
        }
    
    # 数据预处理
    price_df['date'] = pd.to_datetime(price_df['date'])
    weather_df['date'] = pd.to_datetime(weather_df['date'])
    
    # 按日期聚合价格
    price_by_date = price_df.groupby('date')['avg_price'].mean().reset_index()
    
    # 合并价格和天气数据
    merged_df = pd.merge(price_by_date, weather_df, on='date', how='inner')
    
    if merged_df.empty:
        return {
            'data': [],
            'layout': {
                'title': f'没有找到{location}的匹配数据',
                'xaxis': {'title': '日期'},
                'yaxis': {'title': '数值'}
            }
        }
    
    # 计算相关系数
    temp_corr, temp_p = pearsonr(merged_df['avg_price'], merged_df['avg_temp'])
    precip_corr, precip_p = pearsonr(merged_df['avg_price'], merged_df['avg_precip'])
    
    # 创建图表
    fig = go.Figure()
    
    # 添加价格线
    fig.add_trace(go.Scatter(
        x=merged_df['date'],
        y=merged_df['avg_price'],
        mode='lines',
        name='价格',
        line=dict(color='blue', width=3),
        yaxis='y'
    ))
    
    # 添加温度线
    fig.add_trace(go.Scatter(
        x=merged_df['date'],
        y=merged_df['avg_temp'],
        mode='lines',
        name='温度',
        line=dict(color='red', width=2),
        yaxis='y2'
    ))
    
    # 添加降水量柱状图
    fig.add_trace(go.Bar(
        x=merged_df['date'],
        y=merged_df['avg_precip'],
        name='降水量',
        marker_color='skyblue',
        opacity=0.7,
        yaxis='y3'
    ))
    
    # 更新布局
    fig.update_layout(
        title=f'{location} 天气与{herb_name}价格相关性<br>温度相关系数: {temp_corr:.4f}, 降水量相关系数: {precip_corr:.4f}',
        xaxis=dict(title='日期'),
        yaxis=dict(
            title='价格 (元)',
            titlefont=dict(color='blue'),
            tickfont=dict(color='blue')
        ),
        yaxis2=dict(
            title='温度 (°C)',
            titlefont=dict(color='red'),
            tickfont=dict(color='red'),
            anchor='x',
            overlaying='y',
            side='right'
        ),
        yaxis3=dict(
            title='降水量 (mm)',
            titlefont=dict(color='skyblue'),
            tickfont=dict(color='skyblue'),
            anchor='free',
            overlaying='y',
            side='right',
            position=0.95
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        ),
        margin=dict(l=40, r=80, t=80, b=40)
    )
    
    return fig

# 更新季节性分析图
@app.callback(
    Output('seasonality-graph', 'figure'),
    Input('query-button', 'n_clicks'),
    State('herb-dropdown', 'value'),
    State('spec-dropdown', 'value'),
    State('time-dropdown', 'value')
)
def update_seasonality(n_clicks, herb_name, specification, days):
    if n_clicks == 0 or not herb_name or not specification:
        return {
            'data': [],
            'layout': {
                'title': '请选择药材和规格并点击查询',
                'xaxis': {'title': '月份'},
                'yaxis': {'title': '平均价格 (元)'}
            }
        }
    
    # 使用更长的时间范围进行季节性分析
    analysis_days = max(days, 365)
    df = get_price_data(herb_name, specification, analysis_days)
    
    if df.empty:
        return {
            'data': [],
            'layout': {
                'title': f'没有找到{herb_name} {specification}的价格数据',
                'xaxis': {'title': '月份'},
                'yaxis': {'title': '平均价格 (元)'}
            }
        }
    
    # 数据预处理
    df['date'] = pd.to_datetime(df['date'])
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year
    
    # 按月份聚合
    monthly_avg = df.groupby('month')['avg_price'].mean().reset_index()
    monthly_std = df.groupby('month')['avg_price'].std().reset_index()
    
    # 合并平均值和标准差
    monthly_data = pd.merge(monthly_avg, monthly_std, on='month', suffixes=('', '_std'))
    
    # 创建图表
    fig = go.Figure()
    
    # 添加月度平均价格柱状图
    fig.add_trace(go.Bar(
        x=monthly_data['month'],
        y=monthly_data['avg_price'],
        name='月度平均价格',
        marker_color='lightgreen',
        error_y=dict(
            type='data',
            array=monthly_data['avg_price_std'],
            visible=True
        )
    ))
    
    # 如果有足够的数据，添加年度月度热图
    if len(df['year'].unique()) > 1:
        # 创建透视表
        pivot_df = df.pivot_table(index='month', columns='year', values='avg_price', aggfunc='mean')
        
        # 添加热图作为第二个子图
        fig2 = px.imshow(
            pivot_df,
            labels=dict(x="年份", y="月份", color="价格"),
            x=pivot_df.columns,
            y=pivot_df.index,
            color_continuous_scale="Viridis",
            title=f"{herb_name} {specification} 月度价格热图"
        )
        
        # 更新布局为两个子图
        fig = go.Figure(data=fig.data + fig2.data)
        
        fig.update_layout(
            title=f'{herb_name} {specification} 季节性分析',
            xaxis=dict(
                title='月份',
                tickmode='array',
                tickvals=list(range(1, 13)),
                ticktext=['一月', '二月', '三月', '四月', '五月', '六月', 
                          '七月', '八月', '九月', '十月', '十一月', '十二月']
            ),
            yaxis=dict(title='平均价格 (元)'),
            margin=dict(l=40, r=40, t=60, b=80)
        )
    else:
        # 只有一个图表的布局
        fig.update_layout(
            title=f'{herb_name} {specification} 季节性分析',
            xaxis=dict(
                title='月份',
                tickmode='array',
                tickvals=list(range(1, 13)),
                ticktext=['一月', '二月', '三月', '四月', '五月', '六月', 
                          '七月', '八月', '九月', '十月', '十一月', '十二月']
            ),
            yaxis=dict(title='平均价格 (元)'),
            margin=dict(l=40, r=40, t=60, b=80)
        )
    
    return fig

# 更新价格预测图
@app.callback(
    Output('prediction-graph', 'figure'),
    Input('query-button', 'n_clicks'),
    State('herb-dropdown', 'value'),
    State('spec-dropdown', 'value'),
    State('time-dropdown', 'value')
)
def update_prediction(n_clicks, herb_name, specification, days):
    if n_clicks == 0 or not herb_name or not specification:
        return {
            'data': [],
            'layout': {
                'title': '请选择药材和规格并点击查询',
                'xaxis': {'title': '日期'},
                'yaxis': {'title': '价格 (元)'}
            }
        }
    
    df = get_price_data(herb_name, specification, days)
    
    if df.empty:
        return {
            'data': [],
            'layout': {
                'title': f'没有找到{herb_name} {specification}的价格数据',
                'xaxis': {'title': '日期'},
                'yaxis': {'title': '价格 (元)'}
            }
        }
    
    # 数据预处理
    df['date'] = pd.to_datetime(df['date'])
    
    # 按日期聚合
    overall_avg = df.groupby('date')['avg_price'].mean().reset_index()
    
    # 确保数据按日期排序
    overall_avg = overall_avg.sort_values('date')
    
    # 简单的移动平均预测
    window_size = min(30, len(overall_avg) // 3)
    if window_size < 3:
        window_size = 3
    
    # 计算移动平均
    overall_avg['ma'] = overall_avg['avg_price'].rolling(window=window_size).mean()
    
    # 计算价格变化率
    overall_avg['pct_change'] = overall_avg['avg_price'].pct_change()
    
    # 使用最近的变化率预测未来价格
    recent_changes = overall_avg['pct_change'].dropna().tail(window_size)
    avg_change = recent_changes.mean()
    
    # 预测未来30天
    days_to_predict = 30
    last_date = overall_avg['date'].iloc[-1]
    last_price = overall_avg['avg_price'].iloc[-1]
    
    future_dates = [last_date + timedelta(days=i+1) for i in range(days_to_predict)]
    future_prices = []
    
    current_price = last_price
    for _ in range(days_to_predict):
        current_price = current_price * (1 + avg_change)
        future_prices.append(current_price)
    
    # 创建图表
    fig = go.Figure()
    
    # 添加历史价格
    fig.add_trace(go.Scatter(
        x=overall_avg['date'],
        y=overall_avg['avg_price'],
        mode='lines+markers',
        name='历史价格',
        line=dict(color='blue', width=2)
    ))
    
    # 添加移动平均线
    fig.add_trace(go.Scatter(
        x=overall_avg['date'],
        y=overall_avg['ma'],
        mode='lines',
        name=f'{window_size}天移动平均',
        line=dict(color='green', width=2, dash='dash')
    ))
    
    # 添加预测价格
    fig.add_trace(go.Scatter(
        x=future_dates,
        y=future_prices,
        mode='lines',
        name='预测价格',
        line=dict(color='red', width=3)
    ))
    
    # 添加预测区间
    std_dev = overall_avg['avg_price'].std()
    
    fig.add_trace(go.Scatter(
        x=future_dates + future_dates[::-1],
        y=[p + 1.96 * std_dev for p in future_prices] + [p - 1.96 * std_dev for p in future_prices][::-1],
        fill='toself',
        fillcolor='rgba(255, 0, 0, 0.1)',
        line=dict(color='rgba(255, 0, 0, 0)'),
        name='95% 置信区间'
    ))
    
    fig.update_layout(
        title=f'{herb_name} {specification} 价格预测 (未来30天)',
        xaxis_title='日期',
        yaxis_title='价格 (元)',
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        ),
        margin=dict(l=40, r=40, t=60, b=40)
    )
    
    return fig

# 运行应用
if __name__ == '__main__':
    app.run_server(debug=True, host='0.0.0.0', port=8050)

9. 主程序入口

In [ ]:
import argparse
import os
import sys
from datetime import datetime

# 添加当前目录到路径
sys.path.append(os.path.dirname(os.path.abspath(__file__)))

# 导入分析模块
from price_trend_analysis import analyze_price_trends
from weather_price_correlation import analyze_weather_price_correlation
from news_sentiment_analysis import analyze_news_sentiment
from market_comparison import analyze_market_comparison
from seasonality_analysis import analyze_seasonality
from price_prediction import predict_price
from report_generator import generate_report

def main():
    parser = argparse.ArgumentParser(description='中药材价格分析工具')
    
    # 添加子命令
    subparsers = parser.add_subparsers(dest='command', help='可用命令')
    
    # 价格趋势分析命令
    trend_parser = subparsers.add_parser('trend', help='价格趋势分析')
    trend_parser.add_argument('herb_name', help='药材名称')
    trend_parser.add_argument('specification', help='规格')
    trend_parser.add_argument('--days', type=int, default=90, help='分析天数 (默认: 90)')
    
    # 天气相关性分析命令
    weather_parser = subparsers.add_parser('weather', help='天气与价格相关性分析')
    weather_parser.add_argument('herb_name', help='药材名称')
    weather_parser.add_argument('specification', help='规格')
    weather_parser.add_argument('--location', help='地区 (可选)')
    weather_parser.add_argument('--days', type=int, default=180, help='分析天数 (默认: 180)')
    
    # 新闻情感分析命令
    news_parser = subparsers.add_parser('news', help='新闻情感分析')
    news_parser.add_argument('herb_name', help='药材名称')
    news_parser.add_argument('--days', type=int, default=30, help='分析天数 (默认: 30)')
    
    # 市场比较分析命令
    market_parser = subparsers.add_parser('market', help='市场比较分析')
    market_parser.add_argument('herb_name', help='药材名称')
    market_parser.add_argument('specification', help='规格')
    market_parser.add_argument('--days', type=int, default=90, help='分析天数 (默认: 90)')
    
    # 季节性分析命令
    season_parser = subparsers.add_parser('season', help='季节性分析')
    season_parser.add_argument('herb_name', help='药材名称')
    season_parser.add_argument('specification', help='规格')
    season_parser.add_argument('--years', type=int, default=3, help='分析年数 (默认: 3)')
    
    # 价格预测命令
    predict_parser = subparsers.add_parser('predict', help='价格预测')
    predict_parser.add_argument('herb_name', help='药材名称')
    predict_parser.add_argument('specification', help='规格')
    predict_parser.add_argument('--days', type=int, default=90, help='历史数据天数 (默认: 90)')
    predict_parser.add_argument('--future', type=int, default=30, help='预测未来天数 (默认: 30)')
    
    # 生成报告命令
    report_parser = subparsers.add_parser('report', help='生成综合分析报告')
    report_parser.add_argument('herb_name', help='药材名称')
    report_parser.add_argument('specification', help='规格')
    report_parser.add_argument('--days', type=int, default=90, help='分析天数 (默认: 90)')
    
    # 启动仪表板命令
    dashboard_parser = subparsers.add_parser('dashboard', help='启动数据可视化仪表板')
    
    # 解析命令行参数
    args = parser.parse_args()
    
    # 如果没有指定命令，显示帮助信息
    if not args.command:
        parser.print_help()
        return
    
    # 执行相应的命令
    if args.command == 'trend':
        analyze_price_trends(args.herb_name, args.specification, args.days)
    
    elif args.command == 'weather':
        analyze_weather_price_correlation(args.herb_name, args.specification, args.location, args.days)
    
    elif args.command == 'news':
        analyze_news_sentiment(args.herb_name, args.days)
    
    elif args.command == 'market':
        analyze_market_comparison(args.herb_name, args.specification, args.days)
    
    elif args.command == 'season':
        analyze_seasonality(args.herb_name, args.specification, args.years)
    
    elif args.command == 'predict':
        predict_price(args.herb_name, args.specification, args.days, args.future)
    
    elif args.command == 'report':
        generate_report(args.herb_name, args.specification, args.days)
    
    elif args.command == 'dashboard':
        # 导入仪表板模块
        from dashboard import app
        print("启动数据可视化仪表板...")
        app.run_server(debug=True, host='0.0.0.0', port=8050)

if __name__ == "__main__":
    main()

10. 配置文件

In [ ]:
"""
配置文件 - 存储系统配置参数
"""

# 数据库配置
DB_CONFIG = {
    'host': 'localhost',
    'database': 'herb_price_db',
    'user': 'postgres',
    'password': 'your_password',
    'port': 5432
}

# 数据爬取配置
SCRAPING_CONFIG = {
    'user_agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'request_timeout': 30,
    'retry_times': 3,
    'retry_delay': 5
}

# 天气API配置
WEATHER_API_CONFIG = {
    'api_key': 'your_api_key',
    'base_url': 'https://api.weatherapi.com/v1',
    'request_timeout': 10
}

# 新闻API配置
NEWS_API_CONFIG = {
    'api_key': 'your_api_key',
    'base_url': 'https://newsapi.org/v2',
    'request_timeout': 10
}

# 分析配置
ANALYSIS_CONFIG = {
    'default_days': 90,
    'default_years': 3,
    'prediction_days': 30,
    'correlation_threshold': 0.5,
    'significance_level': 0.05
}

# 图表配置
CHART_CONFIG = {
    'default_figsize': (12, 8),
    'dpi': 100,
    'line_width': 2,
    'marker_size': 6,
    'color_palette': 'viridis',
    'grid': True,
    'save_format': 'png'
}

# 报告配置
REPORT_CONFIG = {
    'output_dir': 'reports',
    'template_dir': 'templates',
    'default_template': 'default_report.html',
    'auto_open': True
}

# 仪表板配置
DASHBOARD_CONFIG = {
    'host': '0.0.0.0',
    'port': 8050,
    'debug': True,
    'theme': 'light'
}

# 日志配置
LOG_CONFIG = {
    'log_dir': 'logs',
    'log_level': 'INFO',
    'log_format': '%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    'log_file': 'herb_price_analysis.log',
    'max_log_size': 10 * 1024 * 1024,  # 10MB
    'backup_count': 5
}

# 药材列表
COMMON_HERBS = [
    '当归', '党参', '黄芪', '白术', '茯苓', 
    '川芎', '白芍', '熟地黄', '生地黄', '陈皮',
    '甘草', '枸杞子', '山药', '红参', '黄连',
    '金银花', '连翘', '板蓝根', '柴胡', '丹参'
]

# 规格列表
COMMON_SPECIFICATIONS = [
    '统', '一级', '二级', '三级', '特级',
    '小统', '大统', '选货', '精选', '统货'
]

# 主要市场
MAIN_MARKETS = [
    '安国', '亳州', '成都', '玉林', '广州',
    '昆明', '西安', '郑州', '重庆', '上海'
]

11. 安装脚本

In [ ]:
from setuptools import setup, find_packages

setup(
    name="herb_price_analysis",
    version="0.1.0",
    packages=find_packages(),
    install_requires=[
        "pandas>=1.3.0",
        "numpy>=1.20.0",
        "matplotlib>=3.4.0",
        "seaborn>=0.11.0",
        "scikit-learn>=0.24.0",
        "statsmodels>=0.12.0",
        "psycopg2-binary>=2.9.0",
        "dash>=2.0.0",
        "plotly>=5.0.0",
        "jinja2>=3.0.0",
        "requests>=2.25.0",
        "beautifulsoup4>=4.9.0",
        "nltk>=3.6.0",
        "textblob>=0.15.0",
        "scipy>=1.7.0",
        "prophet>=1.0.0",
    ],
    entry_points={
        "console_scripts": [
            "herb-analysis=anticipate.main:main",
        ],
    },
    author="Your Name",
    author_email="your.email@example.com",
    description="中药材价格分析系统",
    keywords="中药材, 价格分析, 数据可视化",
    url="https://github.com/yourusername/herb_price_analysis",
    classifiers=[
        "Development Status :: 3 - Alpha",
        "Intended Audience :: Science/Research",
        "Programming Language :: Python :: 3",
        "Programming Language :: Python :: 3.8",
        "Programming Language :: Python :: 3.9",
    ],
    python_requires=">=3.8",
)

12. 使用说明文档

# 中药材价格分析系统

这是一个用于分析中药材价格趋势、季节性变化、市场比较和价格预测的系统。

## 功能特点

- **价格趋势分析**：分析中药材价格的历史趋势和波动性
- **天气相关性分析**：研究天气因素与中药材价格的相关性
- **新闻情感分析**：分析新闻报道对中药材价格的影响
- **市场比较分析**：比较不同市场的中药材价格差异
- **季节性分析**：识别中药材价格的季节性模式
- **价格预测**：基于历史数据预测未来价格走势
- **综合分析报告**：生成包含多维度分析的综合报告
- **数据可视化仪表板**：交互式数据可视化界面

## 安装

### 系统要求

- Python 3.8 或更高版本
- PostgreSQL 数据库

### 安装步骤

1. 克隆仓库：

```bash
git clone https://github.com/yourusername/herb_price_analysis.git
cd herb_price_analysis
2. 安装依赖：
```bash
pip install -e .
 ```

或者：

```bash
pip install -r requirements.txt
 ```

3. 配置数据库：
在 config.py 文件中修改数据库配置：

```python
DB_CONFIG = {
    'host': 'localhost',
    'database': 'herb_price_db',
    'user': 'your_username',
    'password': 'your_password',
    'port': 5432
}
 ```

4. 初始化数据库：
```bash
python scripts/init_db.py
 ```

## 使用方法
### 命令行工具
系统提供了命令行工具进行各种分析：

1. 价格趋势分析：
```bash
python -m anticipate.main trend 当归 统 --days 90
 ```

2. 天气相关性分析：
```bash
python -m anticipate.main weather 当归 统 --location 安国 --days 180
 ```
```

3. 新闻情感分析：
```bash
python -m anticipate.main news 当归 --days 30
 ```

4. 市场比较分析：
```bash
python -m anticipate.main market 当归 统 --days 90
 ```

5. 季节性分析：
```bash
python -m anticipate.main season 当归 统 --years 3
 ```

6. 价格预测：
```bash
python -m anticipate.main predict 当归 统 --days 90 --future 30
 ```

7. 生成综合分析报告：
```bash
python -m anticipate.main report 当归 统 --days 90
 ```

8. 启动数据可视化仪表板：
```bash
python -m anticipate.main dashboard
 ```

然后在浏览器中访问 http://localhost:8050

### 数据可视化仪表板
数据可视化仪表板提供了交互式界面，可以：

- 选择不同的药材和规格
- 调整分析时间范围
- 查看价格趋势图
- 比较不同市场的价格
- 分析天气与价格的相关性
- 查看季节性变化
- 预测未来价格走势
## 项目结构
herb_price_analysis/
├── anticipate/                  # 主要代码目录
│   ├── __init__.py
│   ├── price_trend_analysis.py  # 价格趋势分析
│   ├── weather_price_correlation.py  # 天气相关性分析
│   ├── news_sentiment_analysis.py  # 新闻情感分析
│   ├── market_comparison.py     # 市场比较分析
│   ├── seasonality_analysis.py  # 季节性分析
│   ├── price_prediction.py      # 价格预测
│   ├── report_generator.py      # 报告生成器
│   ├── dashboard.py             # 数据可视化仪表板
│   ├── config.py                # 配置文件
│   └── main.py                  # 主程序入口
├── data/                        # 数据目录
│   ├── raw/                     # 原始数据
│   └── processed/               # 处理后的数据
├── scripts/                     # 脚本目录
│   ├── init_db.py               # 数据库初始化脚本
│   └── data_scraper.py          # 数据爬取脚本
├── reports/                     # 报告输出目录
├── logs/                        # 日志目录
├── tests/                       # 测试目录
├── setup.py                     # 安装脚本
├── requirements.txt             # 依赖列表
└── README.md                    # 说明文档

## 数据来源
系统使用以下数据来源：

- 中药材价格数据：来自各大中药材交易市场
- 天气数据：来自气象数据API
- 新闻数据：来自新闻API和网络爬虫
## 贡献
欢迎贡献代码、报告问题或提出改进建议。请遵循以下步骤：

1. Fork 仓库
2. 创建特性分支 ( git checkout -b feature/amazing-feature )
3. 提交更改 ( git commit -m 'Add some amazing feature' )
4. 推送到分支 ( git push origin feature/amazing-feature )
5. 创建 Pull Request
## 许可证
本项目采用 MIT 许可证 - 详情请参阅 LICENSE 文件。

## 联系方式
如有任何问题或建议，请联系： your.email@example.com